# Введение в MapReduce модель на Python


In [1]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [ ]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [ ]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [ ]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [ ]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [ ]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [ ]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [ ]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [ ]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [ ]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [ ]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [ ]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [ ]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, 2.905589809636405),
 (1, 2.905589809636405),
 (2, 2.905589809636405),
 (3, 2.905589809636405),
 (4, 2.905589809636405)]

## Inverted index

In [ ]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('what', ['0', '1']),
 ('is', ['0', '1', '2']),
 ('it', ['0', '1', '2']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [ ]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [ ]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('a', 2), ('banana', 2), ('is', 18), ('it', 18), ('what', 10)]),
 (1, [])]

## TeraSort

In [ ]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, 0.0059671639991895065),
   (None, 0.07724245496172),
   (None, 0.08440804135613444),
   (None, 0.13575647907181598),
   (None, 0.14404826813474803),
   (None, 0.21204275967955666),
   (None, 0.21869633101751806),
   (None, 0.25055756276216923),
   (None, 0.28642389538931257),
   (None, 0.3834487515438496),
   (None, 0.3913614390023946),
   (None, 0.4041378102237341),
   (None, 0.41854626274930695),
   (None, 0.4704310153549396),
   (None, 0.4776995227348928),
   (None, 0.48992216726013693)]),
 (1,
  [(None, 0.5005353544023048),
   (None, 0.5135664686748047),
   (None, 0.53391984417089),
   (None, 0.5587932025401512),
   (None, 0.5673804905854288),
   (None, 0.6926646597910275),
   (None, 0.7237444251339501),
   (None, 0.7557883138083207),
   (None, 0.785709769245918),
   (None, 0.7937098630029404),
   (None, 0.7942646850708935),
   (None, 0.9160468126494941),
   (None, 0.9618068292060864),
   (None, 0.9820764489731459)])]

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [ ]:
input_numbers = [1, 5, 2, 8, 3]

def RECORDREADER():
  for index, number in enumerate(input_numbers):
    yield (index, number)

def MAP(k1: int, v1: int):
  # All numbers contribute to finding the single maximum, so use a common key.
  yield ('max_val', v1)

def REDUCE(key: str, values: Iterator[int]):
  max_number = -float('inf') # Initialize with negative infinity
  for number in values:
    if number > max_number:
      max_number = number
  yield (key, max_number)

def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

# Execute the MapReduce job
output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
print(output)

[('max_val', 8)]


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [ ]:
input_numbers = [1, 5, 2, 8, 3]

def RECORDREADER():
  for index, number in enumerate(input_numbers):
    yield (index, number)

def MAP(k1: int, v1: int):
  # All numbers contribute to finding the single maximum, so use a common key.
  yield ('avg', v1)

def REDUCE(key: str, values: Iterator[int]):
  sum = 0
  for number in values:
    sum += number
  x = sum/len(input_numbers)
  yield (key, x)

def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

# Execute the MapReduce job
output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
print(output)

[('avg', 3.8)]


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [ ]:
def groupbykey(pairs):
    # 1. Сортировка по ключу
    sorted_pairs = sorted(pairs, key=lambda x: x[0])

    result = []
    current_key = None
    current_values = []

    # 2. Группировка
    for key, value in sorted_pairs:
        if key != current_key:
            if current_key is not None:
                result.append((current_key, current_values))
            current_key = key
            current_values = [value]
        else:
            current_values.append(value)

    # 3. Добавляем последнюю группу
    if current_key is not None:
        result.append((current_key, current_values))

    return result

data = [
  ("a", 1),
  ("b", 2),
  ("a", 3),
  ("b", 4),
  ("c", 5),
]

print(groupbykey(data))

[('a', [1, 3]), ('b', [2, 4]), ('c', [5])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [ ]:
input_numbers = [1, 2, 3, 2, 4, 1, 5, 3]

def RECORDREADER():
  for index, number in enumerate(input_numbers):
    yield (index, number)

def MAP(k1: int, v1: int):
  # All numbers contribute to finding the single maximum, so use a common key.
  yield (v1,1)

def REDUCE(key: str, values: Iterator[int]):
  return [key]

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

# Execute the MapReduce job
output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
print(output)

[1, 2, 3, 4, 5]


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [ ]:
R = [
    ("Alice", 25),
    ("Bob", 17),
    ("Charlie", 30),
    ("David", 16),
]

# Предикат
def C(t):
    return t[1] >= 18

def RECORDREADER():
    for index, tuple_value in enumerate(R):
        yield (index, tuple_value)

# MAP: если предикат истинен → (t, t)
def MAP(k1, t):
    if C(t):
        yield (t, t)

# REDUCE: identity
def REDUCE(key, values: Iterator):
    return values   # возвращаем без изменений

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

# Execute
output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[('Alice', 25), ('Charlie', 30)]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [ ]:
R = [
    ("Alice", 25),
    ("Bob", 17),
    ("Charlie", 30),
    ("David", 16),
]

S = [0]

def RECORDREADER():
    for index, tuple_value in enumerate(R):
        yield (index, tuple_value)

def MAP(k1, t):
  t_i = tuple(t[i] for i in S)
  yield (t_i, t_i)

# REDUCE: убираем дубликаты — оставляем одну пару
def REDUCE(key, values: Iterator):
    return [key]   # одна пара (t', t')

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

# Execute
output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[('Alice',), ('Bob',), ('Charlie',), ('David',)]


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [ ]:
R = [
    ("Alice", 25),
    ("Bob", 17),
    ("Charlie", 30),
]

S = [
    ("Charlie", 30),
    ("David", 16),
    ("Alice", 25),
]

def RECORDREADER():
    for index, tuple_value in enumerate(R+S):
        yield (index, tuple_value)

def MAP(k1, t):
  yield (t, t)

# REDUCE: оставляем одну пару (t, t)
def REDUCE(key, values: Iterator):
    return [key]   # один результат на каждый уникальный t

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

# Execute
output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[('Alice', 25), ('Bob', 17), ('Charlie', 30), ('David', 16)]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [ ]:
R = [
    ("Alice", 25),
    ("Bob", 17),
    ("Charlie", 30),
]

S = [
    ("Charlie", 30),
    ("David", 16),
    ("Alice", 25),
]

def RECORDREADER():
    for index, tuple_value in enumerate(R+S):
        yield (index, tuple_value)

def MAP(k1, t):
  yield (t, t)

# REDUCE: если список значений длины >= 2 значит t есть в обоих
def REDUCE(key, values: Iterator):
    values = list(values)
    if len(values) >= 2:
      return [key]
    else:
      return []

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

# Execute
output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[('Alice', 25), ('Charlie', 30)]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [ ]:
R = [
    ("Alice", 25),
    ("Bob", 17),
    ("Charlie", 30),
]

S = [
    ("Charlie", 30),
    ("David", 16),
    ("Alice", 25),
]

def RECORDREADER():
    for t in R:
        yield (None, ("R", t))
    for t in S:
        yield (None, ("S", t))

def MAP(k1, t):
  name, relation = t
  yield (relation, name)

# REDUCE: если значения == ["R"] значит t только в R
def REDUCE(key, values: Iterator):
    values = list(values)
    if values == ['R']:
      return [key]
    else:
      return []

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

# Execute
output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[('Bob', 17)]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [ ]:
R = [
    (1, "x"),
    (2, "y"),
    (3, "x"),
]

S = [
    ("x", 100),
    ("x", 200),
    ("z", 300),
]

def RECORDREADER():
    for a, b in R:
        yield (None, ("R", a, b))
    for b, c in S:
        yield (None, ("S", b, c))

def MAP(k1, t):
  if t[0] == "R":
    _,a,b = t
    yield (b,("R", a))
  else:
    _,b,c = t
    yield (b,("S",c))

def REDUCE(b, values: Iterator):
    values = list(values)
    R_val = [v[1] for v in values if v[0]=="R"]
    S_val = [v[1] for v in values if v[0]=="S"]

    res = []

    for a in R_val:
      for c in S_val:
        res.append((a,b,c))
    return res

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

# Execute
output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[(1, 'x', 100), (1, 'x', 200), (3, 'x', 100), (3, 'x', 200)]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [ ]:
R = [
    ("A", 10, "x"),
    ("A", 20, "y"),
    ("B", 5,  "z"),
    ("B", 15, "w"),
    ("A", 30, "k"),
]

def RECORDREADER():
    for index, tuple_value in enumerate(R):
        yield (index, tuple_value)


def MAP(k1, t):
    a, b, c = t
    yield (a, b)

def make_reduce(theta):
    def REDUCE(key, values: Iterator):
        values = list(values)
        result = theta(values)
        return [(key, result)]
    return REDUCE

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))
REDUCE = make_reduce(sum)
# Execute
output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[('A', 60), ('B', 20)]


#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [5]:
import numpy as np
M = np.array([[1, 2, 3], [4, 5, 6]])
v = np.array([1, 2, 3])
print("numpy result:", np.dot(M,v))

def RECORDREADER():
  for i in range(M.shape[0]):
    for j in range(M.shape[1]):
      yield ("M",i,j,M[i][j])
  for j in range(len(v)):
    yield ("v",j,v[j])

def MAP(*t):
  if t[0]=="M":
    _,i,j,val = t
    yield (j,("M",i,val))
  else:
    _, j,val = t
    yield (j,("v",val))

def REDUCE(key, values:Iterator):
  matrix_vals = []
  vector_val = None
  for v in values:
    if v[0] == "M":
      matrix_vals.append(v)
    else:
      vector_val = v[1]
  if vector_val is not None:
    for _, i,mij in matrix_vals:
      yield (i, mij*vector_val)


# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

partial = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("partial:", partial)


result = {}
for i, val in partial:
    result[i] = result.get(i, 0) + val

print("MapReduce result:", result)

numpy result: [14 32]
partial: [(0, np.int64(1)), (1, np.int64(4)), (0, np.int64(4)), (1, np.int64(10)), (0, np.int64(9)), (1, np.int64(18))]
MapReduce result: {0: np.int64(14), 1: np.int64(32)}


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [6]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [7]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1
  # solution code that yield(k2,v2) pairs

def REDUCE(key, values):
  (i, k) = key
  # solution code that yield(k3,v3) pairs

Проверьте своё решение

In [8]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

TypeError: 'NoneType' object is not iterable

In [9]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

TypeError: 'NoneType' object is not iterable

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [21]:
I, J, K = 2, 3, 10

M = np.random.rand(I, J)
N = np.random.rand(J, K)
reference_solution = np.matmul(M, N)

def RECORDREADER():
    # Потоковая генерация элементов матриц
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            yield ((0, i, j), M[i, j])
    for i in range(N.shape[0]):
        for j in range(N.shape[1]):
            yield ((1, i, j), N[i, j])

def MAP_JOIN(t, val):
    tag, i, j = t
    # Реализация объединения по общему индексу
    if tag == 0:
        yield (j, (tag, i, val))
    else:
        yield (i, (tag, j, val))

def REDUCE_JOIN(join_key, values):
    # Разделение по источникам (M и N)
    part_m = [val for val in values if val[0] == 0]
    part_n = [val for val in values if val[0] == 1]

    for m_val in part_m:
        for n_val in part_n:
            yield ((m_val[1], n_val[1]), m_val[2] * n_val[2])

def MAP_SUM(idx_pair, prod):
    yield (idx_pair, prod)

def REDUCE_SUM(idx_pair, prod_list):
    # Финальная агрегация элементов
    sum_val = 0
    for p in prod_list:
        sum_val += p
    yield (idx_pair, sum_val)

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))


result = MapReduce(RECORDREADER, MAP_JOIN, REDUCE_JOIN)

# Агрегация (сумма произведений)
def INTERMEDIATE_READER():
    return result

final_pairs = MapReduce(INTERMEDIATE_READER, MAP_SUM, REDUCE_SUM)

# проверка
def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

print("Результат верен:", np.allclose(reference_solution, asmatrix(final_pairs)))

Результат верен: True


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [23]:
I, J, K = 2, 3, 10
reducers = 2

M = np.random.rand(I, J)
N = np.random.rand(J, K)
reference_solution = np.matmul(M, N)

def PARTITIONER(key_obj):
    return hash(key_obj) % reducers

def groupbykey_distributed(partitions_list, hash_func):
    shuffled_nodes = [dict() for _ in range(reducers)]
    for chunk in partitions_list:
        for (key, val) in chunk:
            target_idx = hash_func(key)
            node = shuffled_nodes[target_idx]
            if key not in node:
                node[key] = []
            node[key].append(val)
    return [(idx, sorted(node.items())) for (idx, node) in enumerate(shuffled_nodes)]

def MapReduceDistributed(INPUT_SENDER, MAP_OP, REDUCE_OP, PART_FUNC=PARTITIONER, COMBINE_OP=None):
    # Map
    mapped = map(lambda reader: flatten(map(lambda kv: MAP_OP(*kv), reader)), INPUT_SENDER())

    # Локальная агрегация
    if COMBINE_OP is not None:
        mapped = map(lambda shard: flatten(map(lambda kv: COMBINE_OP(*kv), groupbykey(shard))), mapped)

    # Shuffle
    shuffled = groupbykey_distributed(mapped, PART_FUNC)

    # Reduce
    reduced = map(lambda node: (node[0], flatten(map(lambda group: REDUCE_OP(*group), node[1]))), shuffled)

    return reduced

def INPUTFORMAT():
    m_elements = [((0, i, j), M[i, j]) for i in range(I) for j in range(J)]
    yield m_elements

    n_elements = [((1, i, j), N[i, j]) for i in range(J) for j in range(K)]
    yield n_elements

def MAP_JOIN(meta, val):
    m_idx, i, j = meta
    # Ключ для Join — общий индекс j
    if m_idx == 0:
        yield (j, (m_idx, i, val))
    else:
        yield (i, (m_idx, j, val))

def REDUCE_JOIN(shared_key, values):
    left = [v for v in values if v[0] == 0]
    right = [v for v in values if v[0] == 1]
    for a in left:
        for b in right:
            # Выход: ((i, k), m_ij * n_jk)
            yield ((a[1], b[1]), a[2] * b[2])

def MAP_MUL(coords, product):
    yield (coords, product)

def REDUCE_MUL(cell_idx, products):
    # Суммирование всех произведений для ячейки (i, k)
    yield (cell_idx, sum(products))


# Join
distributed_join = MapReduceDistributed(INPUTFORMAT, MAP_JOIN, REDUCE_JOIN)
join_results = [(p_id, list(p_data)) for (p_id, p_data) in distributed_join]

def GET_JOINED():
    for partition in join_results:
        yield partition[1]

# Сложение
distributed_mul = MapReduceDistributed(GET_JOINED, MAP_MUL, REDUCE_MUL)
final_parts = [(p_id, list(p_data)) for (p_id, p_data) in distributed_mul]

# Сборка финального списка
solution = [record for p in final_parts for record in p[1]]

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

print("Результат верен:", np.allclose(reference_solution, asmatrix(solution)))

Результат верен: True


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [28]:
I, J, K = 2, 3, 10
reducers = 2
maps = 3

M = np.random.rand(I, J)
N = np.random.rand(J, K)
reference_solution = np.matmul(M, N)

def PARTITIONER(key):
    return hash(key) % reducers

def groupbykey_distributed(map_outputs, partition_func):
    shards = [dict() for _ in range(reducers)]
    for chunk in map_outputs:
        for k, v in chunk:
            p_idx = partition_func(k)
            target = shards[p_idx]
            if k not in target: target[k] = []
            target[k].append(v)
    return [(i, sorted(s.items())) for i, s in enumerate(shards)]

def MapReduceDistributed(INPUT_GEN, MAP_F, REDUCE_F, PART_F=PARTITIONER, COMBINER=None):
    # Применяем MAP к каждому сплиту от каждого RECORDREADER
    mapped = map(lambda split: flatten(map(lambda kv: MAP_F(*kv), split)), INPUT_GEN())

    if COMBINER:
        mapped = map(lambda m: flatten(map(lambda kv: COMBINER(*kv), groupbykey(m))), mapped)

    # Shuffle
    shuffled = groupbykey_distributed(mapped, PART_F)

    # Reduce
    reduced = map(lambda node: (node[0], flatten(map(lambda g: REDUCE_F(*g), node[1]))), shuffled)
    return reduced

def INPUTFORMAT():
    # Генерируем несколько сплитов для каждой матрицы
    def create_splits(data, n):
        random.shuffle(data) # Перемешивание подтверждает работу со случайными подмножествами
        size = int(np.ceil(len(data) / n))
        for i in range(0, len(data), size):
            yield data[i:i + size]

    m_data = [((0, i, j), M[i, j]) for i in range(I) for j in range(J)]
    n_data = [((1, i, j), N[i, j]) for i in range(J) for j in range(K)]

    import random
    yield from create_splits(m_data, maps)
    yield from create_splits(n_data, maps)

def MAP_JOIN(t, val):
    tag, i, j = t
    if tag == 0: yield (j, (tag, i, val))
    else: yield (i, (tag, j, val))

def REDUCE_JOIN(j, values):
    m_side = [v for v in values if v[0] == 0]
    n_side = [v for v in values if v[0] == 1]
    for m in m_side:
        for n in n_side:
            yield ((m[1], n[1]), m[2] * n[2])

def MAP_SUM(coords, val):
    yield (coords, val)

def REDUCE_SUM(coords, vals):
    yield (coords, sum(vals))


# 1. Join
res_join = MapReduceDistributed(INPUTFORMAT, MAP_JOIN, REDUCE_JOIN)
join_results = [list(p[1]) for p in res_join]

def GET_JOINED():
    for partition in join_results:
        yield partition

# 2. Sum
res_sum = MapReduceDistributed(GET_JOINED, MAP_SUM, REDUCE_SUM)
solution = [record for p in res_sum for record in p[1]]

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat


print("Результат верен:", np.allclose(reference_solution, asmatrix(solution)))

Результат верен: True
